In [1]:
import numpy as np
import scipy
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression as LR, LogisticRegression
import cvxpy as cp
from sklearn.linear_model import Ridge, Lasso
import warnings
warnings.filterwarnings("ignore")
from random import randrange
from sklearn.metrics import mean_squared_error, log_loss
from sklearn.linear_model import HuberRegressor
from sklearn.datasets import load_boston, load_diabetes, load_iris, load_digits, load_breast_cancer
from scipy.special import huber
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

def grad_check_sparse(f, x, analytic_grad, num_checks=12, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

def generate_data(N=100, d=20, sigma=5):
    """ Data for Ridge """
    np.random.seed(1)
    w_star = np.random.randn(d)
    X = np.random.randn(N, d)
    y = X.dot(w_star) + np.random.normal(0, sigma, size=N)
    return X, y

def generate_data_lasso(N=100, d=20, sigma=5, density=0.2):
    """ Data for Lasso """
    np.random.seed(1)
    w_star = np.random.randn(d)
    idxs = np.random.choice(range(d), int((1-density)*d), replace=False)
    for idx in idxs:
        w_star[idx] = 0
    X = np.random.randn(N,d)
    y = X.dot(w_star) + np.random.normal(0, sigma, size=N)
    return X, y

def sigmoid(z):
    return 1/(1 + np.exp(-z))

def generate_data_log_reg(N=50, d=50):
    np.random.seed(1)
    w_star = np.array([1, 0.5, -0.5] + [0]*(d - 3))
    X = (np.random.random((N, d)) - 0.5)*10
    y = np.round(sigmoid(X @ w_star + np.random.randn(N)*0.5))
    return X, y

data = datasets.load_diabetes()
X_train, y_train = data.data, data.target
X_train2, y_train2 = generate_data()
X_train3, y_train3 = generate_data_lasso()
X_train4, y_train4 = generate_data_log_reg()

# Ridge regression - closed form

Write the code for the analytic solution of Ridge regression using only numpy

In [2]:
class RidgeRegression():
    def __init__(self, fit_intercept=True, method="inverse", alpha=1.0):
        self.w = 0
        self.fit_intercept = fit_intercept # bias
        self.method = method
        self.alpha = alpha
    
    def fit(self, X, y):
        ### BEGIN SOLUTION
        _X = X
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
        diag_val = [self.alpha]*_X.shape[1]
        if self.fit_intercept:
            diag_val[0] = 0
        alpha_I = np.diag(diag_val)
        self.w = np.linalg.inv(_X.T @ _X + alpha_I) @ _X.T @ y
        
        ### END SOLUTION
        
    def predict(self, X):
        ### BEGIN SOLUTION
        _X = X
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
            
        return _X @ self.w
        ### END SOLUTION

In [3]:
# without bias
sk_model = Ridge(fit_intercept=False, alpha=0.1)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_train)

model = RidgeRegression(fit_intercept=False, alpha=0.1)
model.fit(X_train, y_train)
pred = model.predict(X_train)

error = rel_error(pred, sk_pred)
print("prediction error inverse: ", error)
assert error <= 1e-11

prediction error inverse:  1.6062539935878552e-13


In [4]:
# with bias
sk_model = Ridge(fit_intercept=True, alpha=0.1)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_train)

model = RidgeRegression(fit_intercept=True, alpha=0.1)
model.fit(X_train, y_train)
pred = model.predict(X_train)

error = rel_error(pred, sk_pred)
print("prediction error inverse: ", error)
assert error <= 1e-11

prediction error inverse:  2.756460321069e-15


# CVXPY

Implement the fit methods using CVXPY

### Linear regression with CVXPY

In [5]:
class LinearRegression():
    def __init__(self, fit_intercept=True):
        self.w = 0
        self.fit_intercept = fit_intercept # bias
    
    def fit(self, X, y):
        ### BEGIN SOLUTION
        _X = X
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
        
        n, d = _X.shape
        w = cp.Variable(d)
        loss = cp.sum_squares(_X @ w - y)
        prob = cp.Problem(cp.Minimize(loss))
        prob.solve()
        self.w = w.value
        ### END SOLUTION
        
    def predict(self, X):
        ### BEGIN SOLUTION
        _X = X
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
            
        return _X @ self.w
        ### END SOLUTION

In [6]:
# Without bias
sk_model = LR(fit_intercept=False)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_train)

model = LinearRegression(fit_intercept=False)
model.fit(X_train, y_train)
pred = model.predict(X_train)

error = rel_error(pred, sk_pred)
print("prediction error: ", error)
assert error <= 1e-11

prediction error:  2.4284147882598e-13


In [7]:
# With bias
sk_model = LR(fit_intercept=True)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_train)

model = LinearRegression(fit_intercept=True)
model.fit(X_train, y_train)
pred = model.predict(X_train)

error = rel_error(pred, sk_pred)
print("prediction error: ", error)
assert error <= 1e-11

prediction error:  1.2123433465911832e-15


### Ridge regression with CVXPY

In [8]:
class RidgeRegressionCVXPY():
    def __init__(self, fit_intercept=True, alpha=1.0):
        self.w = 0
        self.fit_intercept = fit_intercept # bias
        self.alpha = alpha
    
    def fit(self, X, y):
        ### BEGIN SOLUTION
        _X = X
        start_at = 0
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
            start_at = 1
        
        n, d = _X.shape
        w = cp.Variable(d)
        #loss = cp.sum_squares(_X @ w - y) + self.alpha * cp.sum_squares(w[start_at:])
        loss = cp.norm2(_X @ w - y)**2 + self.alpha * cp.norm2(w[start_at:])**2
        prob = cp.Problem(cp.Minimize(loss))
        prob.solve()
        self.w = w.value
        ### END SOLUTION
        
    def predict(self, X):
        ### BEGIN SOLUTION
        _X = X
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
        return _X @ self.w
        ### END SOLUTION

In [9]:
# Without bias
sk_model = Ridge(fit_intercept=False, alpha=0.1)
sk_model.fit(X_train2, y_train2)
sk_pred = sk_model.predict(X_train2)
sk_mse = mean_squared_error(sk_pred, y_train2)

model = RidgeRegressionCVXPY(fit_intercept=False, alpha=0.1)
model.fit(X_train2, y_train2)
pred = model.predict(X_train2)
mse = mean_squared_error(pred, y_train2)

error = rel_error(mse, sk_mse)
print("prediction error: ", error)
assert error <= 1e-8

prediction error:  1.766812153909461e-09


In [10]:
# With bias
sk_model = Ridge(fit_intercept=True, alpha=0.1)
sk_model.fit(X_train2, y_train2)
sk_pred = sk_model.predict(X_train2)
sk_mse = mean_squared_error(sk_pred, y_train2)

model = RidgeRegressionCVXPY(fit_intercept=True, alpha=0.1)
model.fit(X_train2, y_train2)
pred = model.predict(X_train2)
mse = mean_squared_error(pred, y_train2)

error = rel_error(mse, sk_mse)
print("prediction error: ", error)
assert error <= 1e-8

prediction error:  5.1556503641711706e-11


### Lasso with CVXPY

In [11]:
class LassoRegressionCVXPY():
    def __init__(self, fit_intercept=True, alpha=1.0):
        self.w = 0
        self.fit_intercept = fit_intercept # bias
        self.alpha = alpha
    
    def fit(self, X, y):
        ### BEGIN SOLUTION
        _X = X
        start_at = 0
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
            start_at = 1
        
        n, d = _X.shape
        w = cp.Variable(d)
        loss = cp.norm2(_X @ w - y)**2 + self.alpha * cp.norm1(w[start_at:]) 
        prob = cp.Problem(cp.Minimize(loss))
        prob.solve()
        self.w = w.value
        ### END SOLUTION
        
    def predict(self, X):
        ### BEGIN SOLUTION
        _X = X
        if self.fit_intercept:
            _X = np.hstack([np.ones([_X.shape[0],1]), _X])
        return _X @ self.w
        ### END SOLUTION

In [12]:
# Without bias
sk_model = Lasso(fit_intercept=False, alpha=0)
sk_model.fit(X_train3, y_train3)
sk_pred = sk_model.predict(X_train3)
sk_mse = mean_squared_error(sk_pred, y_train3)

model = LassoRegressionCVXPY(fit_intercept=False, alpha=0)
model.fit(X_train3, y_train3)
pred = model.predict(X_train3)
mse = mean_squared_error(pred, y_train3)

error = rel_error(mse, sk_mse)
print("prediction error: ", error)
assert error <= 1e-8

prediction error:  4.287477129514631e-12


In [13]:
# With bias
sk_model = Lasso(fit_intercept=True, alpha=0)
sk_model.fit(X_train3, y_train3)
sk_pred = sk_model.predict(X_train3)
sk_mse = mean_squared_error(sk_pred, y_train3)

model = LassoRegressionCVXPY(fit_intercept=True, alpha=0)
model.fit(X_train3, y_train3)
pred = model.predict(X_train3)
mse = mean_squared_error(pred, y_train3)

error = rel_error(mse, sk_mse)
print("prediction error: ", error)
assert error <= 1e-8

prediction error:  2.1611207126014568e-14


# Gradient descent

### Linear regression with gradient descent

In [14]:
X_train1, y_train1 = X_train, y_train
w1 = np.random.randn(X_train1.shape[1]) * 0.0001
b1 = np.random.randn(1) * 0.0001

In [15]:
def mse_loss_naive(w, b, X, y, alpha=0):
    """
    MSE loss function WITH FOR LOOPs
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    - gradient with respect to bias b
    """
    loss = 0.0
    dw = np.zeros_like(w)
    db = 0.0
    
    ### BEGIN SOLUTION
    N, d = X.shape
    for i in range(N):
        loss += np.square(X[i] @ w + b - y[i]) 
        dw += X[i] * (X[i] @ w + b - y[i])
        db +=  X[i] @ w + b  - y[i]
        
    loss = loss / N + alpha * np.sum(w * w)
    dw = 2 * (dw / N + alpha * w)
    db = 2 * db / N
    ### END SOLUTION
    
    return loss, dw, np.array(db).reshape(1,)

In [16]:
# Without alpha

loss, dw1, db1 = mse_loss_naive(w1, b1, X_train1, y_train1, alpha=0)

sk_loss = mean_squared_error(X_train1 @ w1 + b1, y_train1)
print("Loss error : ",rel_error(loss, sk_loss))
assert rel_error(loss, sk_loss) < 1e-9

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: mse_loss_naive(w1, b1, X_train1, y_train1, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15,  error=1e-5)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: mse_loss_naive(w1, b1, X_train1, y_train1, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15,  error=1e-5)

Loss error :  6.25630685071755e-17
Gradient check w
numerical: 2.892061 analytic: 2.892060, relative error: 2.084097e-07
numerical: 2.892061 analytic: 2.892060, relative error: 2.084097e-07
numerical: -1.376394 analytic: -1.376394, relative error: 1.720778e-07
numerical: -1.553189 analytic: -1.553188, relative error: 2.951001e-07
numerical: -0.315454 analytic: -0.315454, relative error: 4.359265e-07
numerical: -1.376394 analytic: -1.376394, relative error: 1.720778e-07
numerical: -3.234110 analytic: -3.234110, relative error: 1.704282e-08
numerical: -4.145418 analytic: -4.145418, relative error: 4.508240e-08
numerical: -0.315454 analytic: -0.315454, relative error: 4.359265e-07
numerical: -3.153317 analytic: -3.153317, relative error: 4.548242e-08
numerical: -1.275044 analytic: -1.275044, relative error: 1.051587e-07
numerical: -1.553189 analytic: -1.553188, relative error: 2.951001e-07
numerical: -2.801915 analytic: -2.801913, relative error: 3.126504e-07
numerical: -2.801915 analytic

In [17]:
# With alpha

loss, dw1, db1 = mse_loss_naive(w1, b1, X_train1, y_train1, alpha=1)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: mse_loss_naive(w1, b1, X_train1, y_train1, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15,  error=1e-5)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: mse_loss_naive(w1, b1, X_train1, y_train1, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15,  error=1e-5)

Gradient check w
numerical: -2.801732 analytic: -2.801730, relative error: 2.888374e-07
numerical: -1.274870 analytic: -1.274870, relative error: 1.409368e-07
numerical: -4.145255 analytic: -4.145255, relative error: 4.206101e-08
numerical: 2.892035 analytic: 2.892034, relative error: 2.052604e-07
numerical: -4.145255 analytic: -4.145255, relative error: 4.206101e-08
numerical: -0.315509 analytic: -0.315509, relative error: 2.729051e-07
numerical: -2.801732 analytic: -2.801730, relative error: 2.888374e-07
numerical: -4.296156 analytic: -4.296158, relative error: 2.097365e-07
numerical: -2.801732 analytic: -2.801730, relative error: 2.888374e-07
numerical: -4.145255 analytic: -4.145255, relative error: 4.206101e-08
numerical: -4.296156 analytic: -4.296158, relative error: 2.097365e-07
numerical: -4.145255 analytic: -4.145255, relative error: 4.206101e-08
numerical: -1.376493 analytic: -1.376494, relative error: 2.084882e-07
numerical: 2.892035 analytic: 2.892034, relative error: 2.0526

In [18]:
def mse_loss_vectorized(w, b, X, y, alpha=0):
    """
    MSE loss function WITHOUT FOR LOOPs
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    - gradient with respect to bias b
    """
    loss = 0.0
    dw = np.zeros_like(w)
    
    ### BEGIN SOLUTION
    loss = np.mean(np.square(X @ w + b - y))  + alpha * np.sum(w * w)
    dw = 2 * ((X.T @ (X @ w + b - y)) / X.shape[0] +  alpha * w)
    db = 2 * np.sum(X @ w + b - y) / X.shape[0]
    ### END SOLUTION
    
    return loss, dw, np.array(db).reshape(1,)

In [19]:
# Without alpha

loss, dw1, db1 = mse_loss_vectorized(w1, b1, X_train1, y_train1, alpha=0)

sk_loss = mean_squared_error(X_train1 @ w1 + b1, y_train1)
print("Loss error : ",rel_error(loss, sk_loss))
assert rel_error(loss, sk_loss) < 1e-9

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: mse_loss_vectorized(w1, b1, X_train1, y_train1, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15,  error=1e-5)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: mse_loss_vectorized(w1, b1, X_train1, y_train1, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15,  error=1e-5)

Loss error :  0.0
Gradient check w
numerical: -4.296088 analytic: -4.296087, relative error: 1.992399e-08
numerical: -1.275044 analytic: -1.275044, relative error: 3.750226e-08
numerical: -3.153317 analytic: -3.153317, relative error: 1.220253e-08
numerical: -3.234110 analytic: -3.234110, relative error: 3.920108e-08
numerical: 2.892060 analytic: 2.892060, relative error: 1.172617e-08
numerical: -4.296088 analytic: -4.296087, relative error: 1.992399e-08
numerical: -0.315454 analytic: -0.315454, relative error: 4.359265e-07
numerical: -4.296088 analytic: -4.296087, relative error: 1.992399e-08
numerical: 2.892060 analytic: 2.892060, relative error: 1.172617e-08
numerical: -1.275044 analytic: -1.275044, relative error: 3.750226e-08
numerical: -0.315454 analytic: -0.315454, relative error: 4.359265e-07
numerical: -4.296088 analytic: -4.296087, relative error: 1.992399e-08
numerical: -4.145418 analytic: -4.145418, relative error: 1.202881e-09
numerical: -2.801913 analytic: -2.801913, rela

In [20]:
#with alpha

loss, dw1, db1 = mse_loss_vectorized(w1, b1, X_train1, y_train1, alpha=1)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w1: mse_loss_vectorized(w1, b1, X_train1, y_train1, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w1, dw1, 15,  error=1e-5)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b1: mse_loss_vectorized(w1, b1, X_train1, y_train1, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b1, db1, 15,  error=1e-5)

Gradient check w
numerical: -0.315509 analytic: -0.315509, relative error: 2.729051e-07
numerical: -3.153555 analytic: -3.153555, relative error: 9.509994e-09
numerical: -4.145255 analytic: -4.145255, relative error: 1.820237e-09
numerical: -1.553358 analytic: -1.553358, relative error: 3.797100e-08
numerical: -0.315509 analytic: -0.315509, relative error: 2.729051e-07
numerical: -4.145255 analytic: -4.145255, relative error: 1.820237e-09
numerical: -3.234258 analytic: -3.234258, relative error: 4.894618e-08
numerical: -1.274870 analytic: -1.274870, relative error: 1.743566e-09
numerical: -1.553358 analytic: -1.553358, relative error: 3.797100e-08
numerical: -4.145255 analytic: -4.145255, relative error: 1.820237e-09
numerical: -1.274870 analytic: -1.274870, relative error: 1.743566e-09
numerical: -2.801730 analytic: -2.801730, relative error: 3.578145e-08
numerical: -4.145255 analytic: -4.145255, relative error: 1.820237e-09
numerical: -3.234258 analytic: -3.234258, relative error: 4.

### Logistic regression with gradient descent

In [21]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

data = load_breast_cancer()
X_train_cancer, y_train_cancer = data.data, data.target
w2 = np.random.randn(X_train_cancer.shape[1]) * 0.0001
b2 = np.random.randn(1) * 0.0001

In [22]:
def log_loss_naive(w, b, X, y, alpha=0):
    """
    log loss function WITH FOR LOOPs
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    """
    loss = 0.0
    dw = np.zeros_like(w)
    db = 0.0
    
    ### BEGIN SOLUTION
    N, d = X.shape
    A = sigmoid(X @ w + b)  # compute activation
    for i in range(N):
        loss -= y[i] * np.log(A[i]) + (1 - y[i]) * (np.log(1 - A[i]))
        dw += (A[i] - y[i]) * X[i]
        db += (A[i] - y[i])
    
    loss = loss / N + alpha * np.sum(w * w)
    dw = dw / N + 2 * alpha * w
    db = db / N
    ### END SOLUTION
    
    return loss, dw, np.array(db).reshape(1,)

In [23]:
# Without alpha

y_pred_0 = sigmoid(X_train_cancer @ w2 + b2)
y_pred = np.vstack([1-y_pred_0, y_pred_0]).T
sk_loss = log_loss(y_train_cancer, y_pred)

loss, dw2, db2 = log_loss_naive(w2, b2, X_train_cancer, y_train_cancer, alpha=0)
print("Loss error : ",rel_error(loss, sk_loss))
assert rel_error(loss, sk_loss) < 1e-9

print("Gradient check w")
# Check with numerical gradient w
f = lambda w2: log_loss_naive(w2, b2, X_train_cancer, y_train_cancer, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w2, dw2, 15, error=1e-4)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b2: log_loss_naive(w2, b2, X_train_cancer, y_train_cancer, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b2, db2, 15,  error=1e-5)

Loss error :  5.458389989444605e-16
Gradient check w
numerical: -0.004782 analytic: -0.004782, relative error: 4.276651e-12
numerical: -0.007290 analytic: -0.007290, relative error: 1.679206e-09
numerical: -0.005799 analytic: -0.005799, relative error: 1.987114e-09
numerical: -0.007290 analytic: -0.007290, relative error: 1.679206e-09
numerical: -0.001918 analytic: -0.001918, relative error: 1.542241e-08
numerical: 0.015911 analytic: 0.015911, relative error: 1.105724e-10
numerical: 0.019986 analytic: 0.019986, relative error: 8.550971e-10
numerical: 0.019986 analytic: 0.019986, relative error: 8.550971e-10
numerical: -0.012003 analytic: -0.012003, relative error: 1.401476e-09
numerical: -0.005799 analytic: -0.005799, relative error: 1.987114e-09
numerical: 0.006348 analytic: 0.006348, relative error: 1.880964e-09
numerical: 0.044347 analytic: 0.044347, relative error: 2.893365e-10
numerical: 0.023390 analytic: 0.023390, relative error: 1.063058e-09
numerical: 0.000191 analytic: 0.0001

In [24]:
# With alpha

loss, dw2, db2 = log_loss_naive(w2, b2, X_train_cancer, y_train_cancer, alpha=1)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w2: log_loss_naive(w2, b2, X_train_cancer, y_train_cancer, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w2, dw2, 15, error=1e-4)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b2: log_loss_naive(w2, b2, X_train_cancer, y_train_cancer, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b2, db2, 15,  error=1e-5)

Gradient check w
numerical: 0.033045 analytic: 0.033045, relative error: 2.303337e-08
numerical: -0.007606 analytic: -0.007606, relative error: 1.730804e-09
numerical: -0.005778 analytic: -0.005778, relative error: 2.028447e-09
numerical: 68.723684 analytic: 68.724055, relative error: 2.696756e-06
numerical: -0.114393 analytic: -0.114393, relative error: 2.307935e-10
numerical: -0.011919 analytic: -0.011919, relative error: 1.397144e-09
numerical: -0.114393 analytic: -0.114393, relative error: 2.307935e-10
numerical: 0.874682 analytic: 0.874682, relative error: 2.488391e-07
numerical: 0.316265 analytic: 0.316265, relative error: 1.456874e-10
numerical: -0.000168 analytic: -0.000168, relative error: 3.921092e-08
numerical: -0.931984 analytic: -0.931984, relative error: 3.701736e-09
numerical: -0.007606 analytic: -0.007606, relative error: 1.730804e-09
numerical: -0.114393 analytic: -0.114393, relative error: 2.307935e-10
numerical: 0.033045 analytic: 0.033045, relative error: 2.303337e-

In [25]:
def log_loss_vectorized(w, b,X, y, alpha=0):
    """
    log loss function WITHOUT FOR LOOPs
    
    Returns a tuple of:
    - loss 
    - gradient with respect to weights w
    """
    loss = 0.0
    dw = np.zeros_like(w)
    
    ### BEGIN SOLUTION
    N, d = X.shape
    A = sigmoid(X @ w + b)  # compute activation
    loss = (- 1 / N) * np.sum(y * np.log(A) + (1 - y) * (np.log(1 - A))) + alpha * np.sum(w * w)
    dw = (1 / N) *  (A - y).T @ X +  2 * alpha * w
    db = np.sum(A - y) / N
    ### END SOLUTION
    
    return loss, dw, np.array(db).reshape(1,)

In [26]:
# Without alpha

y_pred_0 = sigmoid(X_train_cancer @ w2 + b2)
y_pred = np.vstack([1-y_pred_0, y_pred_0]).T
sk_loss = log_loss(y_train_cancer, y_pred)

loss, dw2, db2 = log_loss_vectorized(w2, b2, X_train_cancer, y_train_cancer, alpha=0)
print("Loss error : ",rel_error(loss, sk_loss))
assert rel_error(loss, sk_loss) < 1e-9

print("Gradient check w")
# Check with numerical gradient w
f = lambda w2: log_loss_vectorized(w2, b2, X_train_cancer, y_train_cancer, alpha=0)[0]
grad_numerical = grad_check_sparse(f, w2, dw2, 15, error=1e-4)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b2: log_loss_vectorized(w2, b2, X_train_cancer, y_train_cancer, alpha=0)[0]
grad_numerical = grad_check_sparse(f2, b2, db2, 15,  error=1e-5)

Loss error :  0.0
Gradient check w
numerical: -0.000751 analytic: -0.000751, relative error: 1.843214e-09
numerical: -0.013661 analytic: -0.013661, relative error: 9.586570e-12
numerical: 0.000298 analytic: 0.000298, relative error: 2.174557e-09
numerical: 0.874605 analytic: 0.874605, relative error: 2.488765e-07
numerical: 0.874605 analytic: 0.874605, relative error: 2.488765e-07
numerical: 68.723713 analytic: 68.724084, relative error: 2.696755e-06
numerical: -0.007290 analytic: -0.007290, relative error: 1.562272e-10
numerical: 0.000191 analytic: 0.000191, relative error: 1.316377e-09
numerical: 0.000191 analytic: 0.000191, relative error: 1.316377e-09
numerical: -0.931829 analytic: -0.931829, relative error: 3.674083e-09
numerical: -0.859438 analytic: -0.859438, relative error: 1.640851e-09
numerical: 0.006348 analytic: 0.006348, relative error: 1.320369e-10
numerical: 0.015911 analytic: 0.015911, relative error: 1.105736e-10
numerical: -0.006269 analytic: -0.006269, relative error

In [27]:
# With alpha

loss, dw2, db2 = log_loss_vectorized(w2, b2, X_train_cancer, y_train_cancer, alpha=1)

print("Gradient check w")
# Check with numerical gradient w
f = lambda w2: log_loss_vectorized(w2, b2, X_train_cancer, y_train_cancer, alpha=1)[0]
grad_numerical = grad_check_sparse(f, w2, dw2, 15, error=1e-4)

print("Gradient check bias")
# Check with numerical gradient b
f2 = lambda b2: log_loss_vectorized(w2, b2, X_train_cancer, y_train_cancer, alpha=1)[0]
grad_numerical = grad_check_sparse(f2, b2, db2, 15,  error=1e-5)

Gradient check w
numerical: 0.023449 analytic: 0.023449, relative error: 9.737611e-11
numerical: -0.004565 analytic: -0.004565, relative error: 5.058333e-10
numerical: 0.016202 analytic: 0.016202, relative error: 8.225550e-11
numerical: -0.013358 analytic: -0.013358, relative error: 7.595560e-11
numerical: 0.442118 analytic: 0.442118, relative error: 3.070237e-09
numerical: 0.000815 analytic: 0.000815, relative error: 1.626831e-09
numerical: -0.002010 analytic: -0.002010, relative error: 1.281334e-09
numerical: 0.442118 analytic: 0.442118, relative error: 3.070237e-09
numerical: 0.019913 analytic: 0.019913, relative error: 1.723606e-10
numerical: 0.442118 analytic: 0.442118, relative error: 3.070237e-09
numerical: 0.000815 analytic: 0.000815, relative error: 1.626831e-09
numerical: 3.707144 analytic: 3.707145, relative error: 1.098271e-07
numerical: -0.006622 analytic: -0.006622, relative error: 1.313875e-10
numerical: 0.023449 analytic: 0.023449, relative error: 9.737611e-11
numerical

### Multinomial regression with gradient descent

In [28]:
data = load_iris()
X_train_iris, y_train_iris = data.data, data.target

W = np.random.randn(X_train_iris.shape[1], 3) * 0.0001

In [29]:
def softmax_loss_naive(W, X, y, alpha):
    """
    Softmax loss function WITH FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    
    # Initialization
    loss = 0.0
    dW = np.zeros_like(W)
    
    # Tandremo ny numeric instability
    ### BEGIN SOLUTION
    
    num_classes = W.shape[1]
    num_train = X.shape[0]

    for i in range(num_train):
        scores = X[i].dot(W)
        scores -= np.max(scores)
        exp_score = np.exp(scores)
        sum_exp = np.sum(exp_score)
        prob = exp_score / sum_exp
        for j in range(num_classes):
            if j == y[i]:
                dW[:, j] += (prob[j] - 1) * X[i]
            else:
                dW[:, j] += prob[j] * X[i]
        loss += -np.log(prob[y[i]])

    loss /= num_train 
    
    # Add regularization to the loss.
    loss += alpha * np.sum(W * W)
    
    dW /= num_train
    dW += 2 * alpha * W
    ### END SOLUTION

    return loss, dW

In [30]:
# Without alpha

loss, dW = softmax_loss_naive(W, X_train_iris, y_train_iris, 0.0)

f = lambda W: softmax_loss_naive(W, X_train_iris, y_train_iris, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: -0.249296 analytic: -0.249296, relative error: 5.818545e-10
numerical: -0.249296 analytic: -0.249296, relative error: 5.818545e-10
numerical: 0.766590 analytic: 0.766590, relative error: 6.781732e-11
numerical: -0.122563 analytic: -0.122563, relative error: 3.622783e-10
numerical: -0.249296 analytic: -0.249296, relative error: 5.818545e-10
numerical: -0.275775 analytic: -0.275775, relative error: 7.099313e-11
numerical: 0.095354 analytic: 0.095354, relative error: 1.095278e-10
numerical: -0.249296 analytic: -0.249296, relative error: 5.818545e-10
numerical: -0.167906 analytic: -0.167906, relative error: 2.517202e-10
numerical: -0.042408 analytic: -0.042408, relative error: 3.623887e-10
numerical: 0.766590 analytic: 0.766590, relative error: 6.781732e-11
numerical: 0.027209 analytic: 0.027209, relative error: 1.044543e-09


In [31]:
# With alpha

loss, dW = softmax_loss_naive(W, X_train_iris, y_train_iris, 2)

f = lambda W: softmax_loss_naive(W, X_train_iris, y_train_iris, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: -0.249209 analytic: -0.249209, relative error: 5.715217e-10
numerical: -0.599126 analytic: -0.599126, relative error: 6.069300e-11
numerical: 0.318326 analytic: 0.318326, relative error: 3.337996e-11
numerical: -0.042137 analytic: -0.042137, relative error: 3.676301e-10
numerical: -0.599126 analytic: -0.599126, relative error: 6.069300e-11
numerical: 0.026608 analytic: 0.026608, relative error: 1.198395e-09
numerical: -0.168356 analytic: -0.168356, relative error: 2.570539e-10
numerical: 0.281755 analytic: 0.281755, relative error: 3.940515e-10
numerical: -0.122336 analytic: -0.122336, relative error: 3.872866e-10
numerical: -0.042137 analytic: -0.042137, relative error: 3.676301e-10
numerical: -0.599126 analytic: -0.599126, relative error: 6.069300e-11
numerical: 0.318326 analytic: 0.318326, relative error: 3.337996e-11


In [32]:
def softmax_loss_vectorized(W, X, y, alpha, fit_intercept=False):
    """
    Softmax loss function WITHOUT FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    # Initialize the loss and gradient to zero.
    loss = 0.0
    dW = np.zeros_like(W)

    ### BEGIN SOLUTION
    num_classes = W.shape[1]
    num_train = X.shape[0]

    scores = X.dot(W)
    scores -= np.expand_dims(np.max(scores, axis=1), axis=1)
    exp_score = np.exp(scores)
    sum_exp = np.sum(exp_score, axis=1)
    prob = exp_score / np.expand_dims(sum_exp, axis=1)
    correct_class_prob = prob[np.arange(num_train), y]
    loss = np.sum(-np.log(correct_class_prob))
    prob[np.arange(num_train), y] -= 1
    dW = X.T @ prob 

    loss /= num_train
    loss += alpha * np.sum(W * W)

    dW /= num_train
    dW += 2 * alpha * W

    ### END SOLUTION

    return loss, dW

In [33]:
# Without alpha

loss, dW = softmax_loss_vectorized(W, X_train_iris, y_train_iris, 0.0)

f = lambda W: softmax_loss_vectorized(W, X_train_iris, y_train_iris, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: -0.249296 analytic: -0.249296, relative error: 5.150528e-10
numerical: 0.027209 analytic: 0.027209, relative error: 6.365068e-10
numerical: 0.766590 analytic: 0.766590, relative error: 7.505863e-11
numerical: -0.167906 analytic: -0.167906, relative error: 3.178424e-10
numerical: -0.042408 analytic: -0.042408, relative error: 1.612083e-10
numerical: -0.122563 analytic: -0.122563, relative error: 1.358183e-10
numerical: -0.122563 analytic: -0.122563, relative error: 1.358183e-10
numerical: -0.122563 analytic: -0.122563, relative error: 1.358183e-10
numerical: 0.766590 analytic: 0.766590, relative error: 7.505863e-11
numerical: 0.095354 analytic: 0.095354, relative error: 2.259601e-10
numerical: 0.281029 analytic: 0.281029, relative error: 4.660070e-10
numerical: 0.095354 analytic: 0.095354, relative error: 2.259601e-10


In [34]:
# With alpha

loss, dW = softmax_loss_vectorized(W, X_train_iris, y_train_iris, 2)

f = lambda W: softmax_loss_vectorized(W, X_train_iris, y_train_iris, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: 0.318326 analytic: 0.318326, relative error: 1.497006e-12
numerical: -0.042137 analytic: -0.042137, relative error: 1.593316e-10
numerical: -0.249209 analytic: -0.249209, relative error: 5.046966e-10
numerical: 0.318326 analytic: 0.318326, relative error: 1.497006e-12
numerical: 0.281755 analytic: 0.281755, relative error: 4.531572e-10
numerical: -0.122336 analytic: -0.122336, relative error: 1.604065e-10
numerical: -0.168356 analytic: -0.168356, relative error: 3.229992e-10
numerical: 0.318326 analytic: 0.318326, relative error: 1.497006e-12
numerical: -0.275783 analytic: -0.275783, relative error: 1.595089e-11
numerical: -0.275783 analytic: -0.275783, relative error: 1.595089e-11
numerical: -0.599126 analytic: -0.599126, relative error: 1.070199e-10
numerical: -0.168356 analytic: -0.168356, relative error: 3.229992e-10


### Gradient descent for Linear models

In [35]:
class LinearModel():
    def __init__(self):
        self.w = None
        self.b = None

    def train(self, X, y, learning_rate=1e-3, alpha=0, num_iters=100, batch_size=200, verbose=False):
        N, d = X.shape
        
        if self.w is None: # Initialization
            self.w = 0.001 * np.random.randn(d)
            self.b = 0.0

        # Run stochastic gradient descent to optimize w
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            ### BEGIN SOLUTION
            indices = np.random.choice(N, batch_size)
            X_batch = X[indices]
            y_batch = y[indices]
            ### END SOLUTION
            
            # evaluate loss and gradient
            loss, dw, db = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            # perform parameter update                                                                
            # Update the weights w and bias b using the gradient and the learning rate.          
            ### BEGIN SOLUTION
            self.w -= learning_rate*dw
            self.b -= learning_rate*db
            ### END SOLUTION
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class LinearRegressor(LinearModel):
    """ Linear regression """

    def loss(self, X_batch, y_batch, alpha):
        return mse_loss_vectorized(self.w, self.b, X_batch, y_batch, alpha)
    
    def predict(self, X):
        ### BEGIN SOLUTION
        y_pred = X @ self.w + self.b
        return y_pred
        ### END SOLUTION

class LogisticRegressor(LinearModel):
    """ Linear regression """

    def loss(self, X_batch, y_batch, alpha):
        return log_loss_vectorized(self.w, self.b, X_batch, y_batch, alpha)
    
    def predict(self, X):
        """ Return prediction labels vector of 0 or 1 """
        ### BEGIN SOLUTION
        score = X @ self.w + self.b
        prob = sigmoid(score)
        y_pred = prob > 0.5
        return y_pred.astype(int) 
        ### END SOLUTION

In [36]:
# Linear regression with gradient descent

sk_model = LinearRegression(fit_intercept=True)
sk_model.fit(X_train1, y_train1)
sk_pred = sk_model.predict(X_train1)
sk_mse = mean_squared_error(sk_pred, y_train1)

model = LinearRegressor()
model.train(X_train1, y_train1, num_iters=75000, batch_size=64, learning_rate=1e-2, verbose=True)
pred = model.predict(X_train1)
mse = mean_squared_error(pred, y_train1)

print("MSE scikit-learn:", sk_mse)
print("MSE gradient descent model :", mse)
assert mse - sk_mse < 100

iteration 0 / 75000: loss 32800.422018
iteration 10000 / 75000: loss 2986.576939
iteration 20000 / 75000: loss 2800.542471
iteration 30000 / 75000: loss 2660.571294
iteration 40000 / 75000: loss 2161.373254
iteration 50000 / 75000: loss 3989.601371
iteration 60000 / 75000: loss 2728.937092
iteration 70000 / 75000: loss 2545.973784
MSE scikit-learn: 2859.6963475867506
MSE gradient descent model : 2884.8973656858293


In [37]:
# Logistc regression with gradient descent

scaler = StandardScaler()
X_train_cancer = scaler.fit_transform(X_train_cancer)

sk_model = LogisticRegression(fit_intercept=True)
sk_model.fit(X_train_cancer, y_train_cancer)
sk_pred = sk_model.predict(X_train_cancer)
sk_log_loss = log_loss(sk_pred, y_train_cancer)

model = LogisticRegressor()
model.train(X_train_cancer, y_train_cancer, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X_train_cancer)
model_log_loss = log_loss(pred, y_train_cancer)

print("Log-loss scikit-learn:", sk_log_loss)
print("Log-loss gradiet descent model :", model_log_loss)
print("Error :", rel_error(sk_log_loss, model_log_loss))
assert rel_error(sk_log_loss, model_log_loss) < 1e-7

iteration 0 / 75000: loss 0.691996
iteration 10000 / 75000: loss 0.077291
iteration 20000 / 75000: loss 0.090863
iteration 30000 / 75000: loss 0.048366
iteration 40000 / 75000: loss 0.067384
iteration 50000 / 75000: loss 0.085959
iteration 60000 / 75000: loss 0.055705
iteration 70000 / 75000: loss 0.131560
Log-loss scikit-learn: 0.4249086712816093
Log-loss gradiet descent model : 0.4249086712816093
Error : 0.0


### Gradient descent for Multinomial logistc regression

In [38]:
class LinearModel():
    def __init__(self, fit_intercept=True):
        self.W = None
        self.fit_intercept = fit_intercept

    def train(self, X, y, learning_rate=1e-3, alpha=0, num_iters=100, batch_size=200, verbose=False):
        if self.fit_intercept:
            ### BEGIN SOLUTION
            X = np.hstack([np.ones([X.shape[0],1]), X])
            ### END SOLUTION
            
        N, d = X.shape
        
        C = (np.max(y) + 1) 
        if self.W is None: # Initialization
            self.W = 0.001 * np.random.randn(d, C)

        # Run stochastic gradient descent to optimize W
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            ### BEGIN SOLUTION
            indices = np.random.choice(N, batch_size)
            X_batch = X[indices]
            y_batch = y[indices]
            ### END SOLUTION
            
            # evaluate loss and gradient
            loss, dW = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.          
            ### BEGIN SOLUTION
            self.W -= learning_rate*dW
            ### END SOLUTION
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class MultinomialLogisticRegressor(LinearModel):
    """ Softmax regression """

    def loss(self, X_batch, y_batch, alpha):
        return softmax_loss_vectorized(self.W, X_batch, y_batch, alpha)
    
    def predict(self, X):
        """ 
        Inputs:
        - X: array of shape (N, D) 

        Returns:
        - y_pred: 1-dimensional array of length N, each element is an integer giving the predicted class 
        """
        ### BEGIN SOLUTION
        if self.fit_intercept:
            X = np.hstack([np.ones([X.shape[0],1]), X])
            
        y_pred = X @ self.W
        y_pred = np.argmax(y_pred, axis=1)
        ### END SOLUTION
        return y_pred

In [41]:
# Without bias

scaler = StandardScaler()
X_train_iris = scaler.fit_transform(X_train_iris)

sk_model = LogisticRegression(fit_intercept=False)
sk_model.fit(X_train_iris, y_train_iris)
sk_pred = sk_model.predict(X_train_iris)
sk_accuracy = accuracy_score(y_train_iris, sk_pred)

model = MultinomialLogisticRegressor(fit_intercept=False)
model.train(X_train_iris, y_train_iris, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X_train_iris)
model_accuracy = accuracy_score(y_train_iris, pred)

print("Accuracy scikit-learn:", sk_accuracy)
print("Accuracy gradient descent model :", model_accuracy)
assert sk_accuracy - model_accuracy < 0.01

iteration 0 / 75000: loss 1.099491
iteration 10000 / 75000: loss 0.338989
iteration 20000 / 75000: loss 0.347744
iteration 30000 / 75000: loss 0.281512
iteration 40000 / 75000: loss 0.483232
iteration 50000 / 75000: loss 0.413908
iteration 60000 / 75000: loss 0.277603
iteration 70000 / 75000: loss 0.233435
Accuracy scikit-learn: 0.86
Accuracy gradient descent model : 0.8666666666666667


In [43]:
# With bias

scaler = StandardScaler()
X_train_iris = scaler.fit_transform(X_train_iris)

sk_model = LogisticRegression(fit_intercept=True)
sk_model.fit(X_train_iris, y_train_iris)
sk_pred = sk_model.predict(X_train_iris)
sk_accuracy = accuracy_score(y_train_iris, sk_pred)

model = MultinomialLogisticRegressor(fit_intercept=True)
model.train(X_train_iris, y_train_iris, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X_train_iris)
model_accuracy = accuracy_score(y_train_iris, pred)

print("Accuracy scikit-learn:", sk_accuracy)
print("Accuracy gradient descent model :", model_accuracy)
assert sk_accuracy - model_accuracy < 0.02

iteration 0 / 75000: loss 1.098857
iteration 10000 / 75000: loss 0.363482
iteration 20000 / 75000: loss 0.226487
iteration 30000 / 75000: loss 0.191297
iteration 40000 / 75000: loss 0.168036
iteration 50000 / 75000: loss 0.173357
iteration 60000 / 75000: loss 0.120193
iteration 70000 / 75000: loss 0.183873
Accuracy scikit-learn: 0.9733333333333334
Accuracy gradient descent model : 0.96
